In [ ]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

In [ ]:
import cellrank
import scanpy
import anndata
import petsc4py
import slepc4py

from petsc4py import PETSc
from slepc4py import SLEPc

print("CellRank :", cellrank.__version__)
print("Scanpy   :", scanpy.__version__)
print("AnnData  :", anndata.__version__)
print("petsc4py :", petsc4py.__version__)
print("slepc4py :", slepc4py.__version__)
print("PETSc    :", PETSc.Sys.getVersion())
print("SLEPc    :", SLEPc.Sys.getVersion())

print("\nWSL GPCCA ENVIRONMENT: PASS")

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import scanpy as sc
import anndata as ad
import cellrank as cr

STEP28_DIR = Path(
    _hcc_path('LIHC_scRNA_Step28_CellRank')
)

TABLE_DIR = STEP28_DIR / "Tables"
FIG_DIR = STEP28_DIR / "Figures"
DATA_DIR = STEP28_DIR / "Data"
AUDIT_DIR = STEP28_DIR / "Audit"
MANIFEST_DIR = STEP28_DIR / "Manifest"

INPUT_H5AD = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step27_Trajectory/Data/"
    "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
)

for d in [
    TABLE_DIR,
    FIG_DIR,
    DATA_DIR,
    AUDIT_DIR,
    MANIFEST_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Input exists:", INPUT_H5AD.exists())
print("Step28 directory:", STEP28_DIR)

In [ ]:
adata28 = sc.read_h5ad(INPUT_H5AD)

STATE_COL = "final_malignant_state_id"
PATIENT_COL = "patient"
PT_KEY = "MS6_anchored_pseudotime"
CONN_KEY = "neighbors_lineage_clean_connectivities"

assert adata28.n_obs == 12057
assert adata28.n_vars == 25712

assert STATE_COL in adata28.obs.columns
assert PATIENT_COL in adata28.obs.columns
assert PT_KEY in adata28.obs.columns
assert CONN_KEY in adata28.obsp

print("Shape:", adata28.shape)
print("States:", adata28.obs[STATE_COL].nunique())
print(
    "Pseudotime range:",
    float(adata28.obs[PT_KEY].min()),
    float(adata28.obs[PT_KEY].max())
)
print(
    "Frozen graph shape:",
    adata28.obsp[CONN_KEY].shape
)

In [ ]:
from cellrank.kernels import PseudotimeKernel

pk_wsl = PseudotimeKernel(
    adata28,
    time_key=PT_KEY,
    backward=False,
    conn_key=CONN_KEY,
    check_connectivity=False
)

pk_wsl.compute_transition_matrix(
    threshold_scheme="hard",
    frac_to_keep=0.30,
    check_irreducibility=False,
    n_jobs=1,
    show_progress_bar=False
)

T_wsl = pk_wsl.transition_matrix.tocsr()

row_sums = np.asarray(
    T_wsl.sum(axis=1)
).ravel()

print("Shape:", T_wsl.shape)
print("NNZ:", T_wsl.nnz)
print(
    "Max row-sum error:",
    np.max(np.abs(row_sums - 1.0))
)

In [ ]:
pt = adata28.obs[
    PT_KEY
].to_numpy(dtype=float)

coo = T_wsl.tocoo()

src = coo.row
dst = coo.col
w = coo.data

delta = pt[dst] - pt[src]

off = src != dst

forward = (
    off &
    (delta > 1e-12)
)

backward = (
    off &
    (delta < -1e-12)
)

off_total = w[off].sum()

forward_fraction = (
    w[forward].sum() / off_total
)

backward_fraction = (
    w[backward].sum() / off_total
)

print(
    "Forward fraction:",
    forward_fraction
)

print(
    "Backward fraction:",
    backward_fraction
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 7
# Formal Windows-vs-WSL reproducibility check
# =============================================================================

states = [f"MS{i}" for i in range(1, 9)]

state_values = (
    adata28.obs[STATE_COL]
    .astype(str)
    .to_numpy()
)

state_flow_wsl = pd.DataFrame(
    0.0,
    index=states,
    columns=states
)

for source in states:

    src_idx = np.flatnonzero(
        state_values == source
    )

    for target in states:

        dst_idx = np.flatnonzero(
            state_values == target
        )

        state_flow_wsl.loc[
            source, target
        ] = float(
            T_wsl[src_idx][:, dst_idx].sum()
            / len(src_idx)
        )


windows_state = pd.read_csv(
    TABLE_DIR /
    "Step28B_State_Transition_Matrix.csv",
    index_col=0
)

windows_state = (
    windows_state
    .loc[states, states]
    .astype(float)
)

state_diff = (
    state_flow_wsl -
    windows_state
)

max_state_diff = float(
    np.abs(state_diff.values).max()
)


windows_direction = pd.read_csv(
    TABLE_DIR /
    "Step28B_OffDiagonal_Direction_QC.csv",
    index_col=0
)["Value"]

direction_wsl = pd.Series({
    "Forward_fraction":
        forward_fraction,

    "Backward_fraction":
        backward_fraction,

    "Equal_PT_fraction":
        0.0
})

common = [
    "Forward_fraction",
    "Backward_fraction",
    "Equal_PT_fraction"
]

direction_compare = pd.DataFrame({
    "Windows":
        windows_direction.loc[common],

    "WSL":
        direction_wsl.loc[common]
})

direction_compare[
    "Absolute_difference"
] = (
    direction_compare["Windows"]
    - direction_compare["WSL"]
).abs()


CROSS_ENV_PASS = (
    T_wsl.nnz == 164442
    and
    max_state_diff < 1e-10
    and
    direction_compare[
        "Absolute_difference"
    ].max() < 1e-10
)


print(
    "Maximum state-flow difference:",
    max_state_diff
)

display(direction_compare)

print(
    "\nCROSS_ENV_PASS:",
    CROSS_ENV_PASS
)

assert CROSS_ENV_PASS

In [ ]:
# =============================================================================
# Step28D-WSL Cell 8
# Initialize GPCCA
# =============================================================================

assert CROSS_ENV_PASS

from cellrank.estimators import GPCCA

g = GPCCA(
    pk_wsl
)

print(g)

print(
    "Estimator shape:",
    g.shape
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 9
# PETSc/SLEPc partial sparse Schur decomposition
# =============================================================================

g.compute_schur(
    n_components=12,
    method="krylov",
    which="LR"
)

print(
    "Schur decomposition complete."
)

print(
    "Schur vectors shape:",
    g.schur_vectors.shape
)

print(
    "Schur matrix shape:",
    g.schur_matrix.shape
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 10
# Extract leading eigenspectrum
# =============================================================================

eig = g.eigendecomposition

eigvals = np.asarray(
    eig["D"]
)

if eigvals.ndim == 2:
    eigvals = np.diag(
        eigvals
    )

eig_table = pd.DataFrame({
    "Component":
        np.arange(
            1,
            len(eigvals) + 1
        ),

    "Real":
        np.real(eigvals),

    "Imag":
        np.imag(eigvals),

    "Magnitude":
        np.abs(eigvals)
})

display(
    eig_table.round(6)
)

print(
    "Eigengap:",
    eig.get("eigengap")
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 11
# Save sparse-Schur outputs
# =============================================================================

state_flow_wsl.to_csv(
    TABLE_DIR /
    "Step28D_WSL_State_Transition_Matrix.csv"
)

direction_compare.to_csv(
    TABLE_DIR /
    "Step28D_Windows_WSL_Comparison.csv"
)

eig_table.to_csv(
    TABLE_DIR /
    "Step28D_Schur_Eigenvalues.csv",
    index=False
)

print(
    "Step28D sparse-Schur tables saved."
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 12
# REPORT — sparse Schur
# =============================================================================

print("=" * 100)
print("STEP28D-WSL SPARSE SCHUR REPORT")
print("=" * 100)

print(
    "\nPython executable:",
    sys.executable
)

print(
    "\nCells / genes:",
    adata28.n_obs,
    "/",
    adata28.n_vars
)

print(
    "\nTransition matrix:",
    T_wsl.shape
)

print(
    "NNZ:",
    T_wsl.nnz
)

print(
    "\nWindows-vs-WSL "
    "maximum state-flow difference:",
    max_state_diff
)

print(
    "\nForward fraction:",
    forward_fraction
)

print(
    "Backward fraction:",
    backward_fraction
)

print(
    "\nCROSS_ENV_PASS:",
    CROSS_ENV_PASS
)

print(
    "\nSchur vectors:",
    g.schur_vectors.shape
)

print(
    "Schur matrix:",
    g.schur_matrix.shape
)

print(
    "\nLeading eigenspectrum:"
)

display(
    eig_table.round(6)
)

print(
    "\nEigengap:",
    eig.get("eigengap")
)

print(
    "\nFallback status:"
)

print(
    "Expected: PETSc/SLEPc sparse Krylov; "
    "NO brandts / NO densifying."
)

print(
    "\nSTEP28D SPARSE SCHUR COMPLETE"
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 13
# Primary data-driven GPCCA macrostates
# n_states=None -> eigengap heuristic
# =============================================================================

g.compute_macrostates(
    n_states=None,
    n_cells=30,
    cluster_key=None
)

macro_memberships = g.macrostates_memberships
macro_names = list(macro_memberships.names)

print("Number of macrostates:", len(macro_names))
print("Macrostate names:", macro_names)

assert len(macro_names) == 3

print(
    "PASS — primary eigengap-driven "
    "3-macrostate solution computed."
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 14
# Extract soft macrostate memberships
# =============================================================================

M = np.asarray(
    macro_memberships
)

assert M.shape == (
    adata28.n_obs,
    len(macro_names)
)

hard_idx = np.argmax(
    M,
    axis=1
)

hard_macro = np.asarray(
    macro_names,
    dtype=object
)[hard_idx]

max_membership = np.max(
    M,
    axis=1
)

print("Membership matrix:", M.shape)

print(
    "Median max membership:",
    float(np.median(max_membership))
)

print(
    "P10:",
    float(np.quantile(max_membership, 0.10))
)

print(
    "P90:",
    float(np.quantile(max_membership, 0.90))
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 15
# Soft composition by frozen malignant state
# =============================================================================

states = [
    f"MS{i}" for i in range(1, 9)
]

state_values = (
    adata28.obs[STATE_COL]
    .astype(str)
    .to_numpy()
)

macro_state_soft = pd.DataFrame(
    0.0,
    index=macro_names,
    columns=states
)

for j, macro in enumerate(macro_names):

    total_weight = M[:, j].sum()

    for state in states:

        mask = (
            state_values == state
        )

        macro_state_soft.loc[
            macro,
            state
        ] = (
            M[mask, j].sum()
            / total_weight
        )

display(
    macro_state_soft.round(4)
)

print(
    "\nRow sums:"
)

display(
    macro_state_soft
    .sum(axis=1)
    .round(6)
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 16
# Soft composition by patient
# =============================================================================

patient_values = (
    adata28.obs[PATIENT_COL]
    .astype(str)
    .to_numpy()
)

patients = sorted(
    np.unique(patient_values)
)

macro_patient_soft = pd.DataFrame(
    0.0,
    index=macro_names,
    columns=patients
)

for j, macro in enumerate(macro_names):

    total_weight = M[:, j].sum()

    for patient in patients:

        mask = (
            patient_values == patient
        )

        macro_patient_soft.loc[
            macro,
            patient
        ] = (
            M[mask, j].sum()
            / total_weight
        )

display(
    macro_patient_soft.round(4)
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 17
# Macrostate pseudotime position
# =============================================================================

pt = (
    adata28.obs[PT_KEY]
    .to_numpy(dtype=float)
)

macro_pt_rows = []

for j, macro in enumerate(macro_names):

    w = M[:, j]

    hard_mask = (
        hard_macro == macro
    )

    macro_pt_rows.append({

        "Macrostate":
            macro,

        "Soft_weight":
            float(w.sum()),

        "Hard_N":
            int(hard_mask.sum()),

        "Weighted_mean_PT":
            float(
                np.sum(w * pt)
                / np.sum(w)
            ),

        "Hard_median_PT":
            float(
                np.median(
                    pt[hard_mask]
                )
            ),

        "Mean_max_membership":
            float(
                np.mean(
                    max_membership[
                        hard_mask
                    ]
                )
            )
    })

macro_pt = pd.DataFrame(
    macro_pt_rows
)

display(
    macro_pt
    .sort_values("Weighted_mean_PT")
    .round(4)
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 18
# Biological program profile of macrostates
# =============================================================================

program_cols = [
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]

for col in program_cols:
    assert col in adata28.obs.columns

macro_program = pd.DataFrame(
    index=macro_names,
    columns=program_cols,
    dtype=float
)

for j, macro in enumerate(macro_names):

    w = M[:, j]

    for col in program_cols:

        x = (
            adata28.obs[col]
            .to_numpy(dtype=float)
        )

        valid = np.isfinite(x)

        macro_program.loc[
            macro,
            col
        ] = (
            np.sum(
                w[valid] * x[valid]
            )
            /
            np.sum(w[valid])
        )

display(
    macro_program.round(4)
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 19
# GPCCA coarse-grained transition matrix
# =============================================================================

coarse_T = g.coarse_T.copy()

display(
    coarse_T.round(4)
)

print(
    "\nCoarse row sums:"
)

display(
    coarse_T
    .sum(axis=1)
    .to_frame("Row_sum")
    .round(6)
)

print(
    "\nMacrostate stabilities:"
)

coarse_stability = pd.Series(
    np.diag(
        coarse_T.to_numpy()
    ),
    index=coarse_T.index,
    name="Stability"
)

display(
    coarse_stability
    .sort_values(
        ascending=False
    )
    .to_frame()
    .round(4)
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 20
# Integrated macrostate summary
# =============================================================================

summary_rows = []

pt_lookup = (
    macro_pt
    .set_index("Macrostate")
)

for macro in macro_names:

    s = macro_state_soft.loc[macro]
    p = macro_patient_soft.loc[macro]

    summary_rows.append({

        "Macrostate":
            macro,

        "Dominant_state":
            s.idxmax(),

        "Dominant_state_fraction":
            float(s.max()),

        "Dominant_patient":
            p.idxmax(),

        "Dominant_patient_fraction":
            float(p.max()),

        "Weighted_mean_PT":
            float(
                pt_lookup.loc[
                    macro,
                    "Weighted_mean_PT"
                ]
            ),

        "Hard_N":
            int(
                pt_lookup.loc[
                    macro,
                    "Hard_N"
                ]
            ),

        "Stability":
            float(
                coarse_stability.loc[
                    macro
                ]
            )
    })

macro_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("Weighted_mean_PT")
    .reset_index(drop=True)
)

display(
    macro_summary.round(4)
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 21
# Save primary GPCCA macrostate outputs
# =============================================================================

macro_state_soft.to_csv(
    TABLE_DIR /
    "Step28D_Primary3_State_Composition.csv"
)

macro_patient_soft.to_csv(
    TABLE_DIR /
    "Step28D_Primary3_Patient_Composition.csv"
)

macro_pt.to_csv(
    TABLE_DIR /
    "Step28D_Primary3_Pseudotime_Position.csv",
    index=False
)

macro_program.to_csv(
    TABLE_DIR /
    "Step28D_Primary3_Program_Profile.csv"
)

coarse_T.to_csv(
    TABLE_DIR /
    "Step28D_Primary3_Coarse_Transition_Matrix.csv"
)

macro_summary.to_csv(
    TABLE_DIR /
    "Step28D_Primary3_Macrostate_Summary.csv",
    index=False
)

print(
    "Primary 3-macrostate tables saved."
)

In [ ]:
# =============================================================================
# Step28D-WSL Cell 22
# REPORT — Primary GPCCA macrostates
# =============================================================================

print("=" * 100)
print("STEP28D PRIMARY GPCCA MACROSTATE REPORT")
print("=" * 100)

print(
    "\nPrimary rule:"
)

print(
    "n_states=None -> eigengap heuristic"
)

print(
    "Eigengap:",
    g.eigendecomposition["eigengap"]
)

print(
    "Number of macrostates:",
    len(macro_names)
)

print(
    "\n[1] Macrostate summary"
)

display(
    macro_summary.round(4)
)

print(
    "\n[2] Frozen-state soft composition"
)

display(
    macro_state_soft.round(4)
)

print(
    "\n[3] Patient soft composition"
)

display(
    macro_patient_soft.round(4)
)

print(
    "\n[4] Biological program profile"
)

display(
    macro_program.round(4)
)

print(
    "\n[5] Coarse transition matrix"
)

display(
    coarse_T.round(4)
)

print(
    "\n[6] Membership confidence"
)

print(
    "Median:",
    round(
        float(
            np.median(max_membership)
        ),
        4
    )
)

print(
    "P10:",
    round(
        float(
            np.quantile(
                max_membership,
                0.10
            )
        ),
        4
    )
)

print(
    "P90:",
    round(
        float(
            np.quantile(
                max_membership,
                0.90
            )
        ),
        4
    )
)

print(
    "\nInterpretation boundary:"
)

print(
    "Macrostates describe metastable structure "
    "of a Step27-pseudotime-conditioned transition matrix."
)

print(
    "They are NOT independent proof of temporal "
    "lineage conversion."
)

print(
    "\nSTEP28D PRIMARY GPCCA COMPLETE"
)

In [ ]:
# =============================================================================
# Step28E Cell 23
# Global macrostate resolution sensitivity
# =============================================================================

from cellrank.estimators import GPCCA

g_res = GPCCA(pk_wsl)

g_res.compute_schur(
    n_components=12,
    method="krylov",
    which="LR"
)

print(
    "Schur vectors:",
    g_res.schur_vectors.shape
)

In [ ]:
# =============================================================================
# Step28E Cell 24
# minChi-selected macrostate resolution
# =============================================================================

candidate_range = (3, 8)

g_res.compute_macrostates(
    n_states=candidate_range,
    n_cells=30,
    cluster_key=None
)

res_memberships = (
    g_res.macrostates_memberships
)

res_names = list(
    res_memberships.names
)

selected_n_states = len(
    res_names
)

print(
    "Candidate resolutions:",
    candidate_states
)

print(
    "minChi-selected number:",
    selected_n_states
)

print(
    "Macrostate names:",
    res_names
)

In [ ]:
# =============================================================================
# Step28E Cell 25
# Extract selected-resolution soft memberships
# =============================================================================

M_res = np.asarray(
    res_memberships
)

assert M_res.shape == (
    adata28.n_obs,
    selected_n_states
)

hard_idx_res = np.argmax(
    M_res,
    axis=1
)

hard_macro_res = np.asarray(
    res_names,
    dtype=object
)[hard_idx_res]

max_membership_res = np.max(
    M_res,
    axis=1
)

print(
    "Membership shape:",
    M_res.shape
)

print(
    "Median confidence:",
    float(
        np.median(
            max_membership_res
        )
    )
)

print(
    "P10 confidence:",
    float(
        np.quantile(
            max_membership_res,
            0.10
        )
    )
)

print(
    "P90 confidence:",
    float(
        np.quantile(
            max_membership_res,
            0.90
        )
    )
)

In [ ]:
# =============================================================================
# Step28E Cell 26
# Frozen malignant-state composition
# =============================================================================

state_comp_res = pd.DataFrame(
    0.0,
    index=res_names,
    columns=states
)

for j, macro in enumerate(
    res_names
):

    total_weight = M_res[:, j].sum()

    for state in states:

        mask = (
            state_values == state
        )

        state_comp_res.loc[
            macro,
            state
        ] = (
            M_res[mask, j].sum()
            / total_weight
        )

display(
    state_comp_res.round(4)
)

print(
    "\nRow sums:"
)

display(
    state_comp_res
    .sum(axis=1)
    .to_frame("Row_sum")
    .round(6)
)

In [ ]:
# =============================================================================
# Step28E Cell 27
# Patient composition
# =============================================================================

patient_comp_res = pd.DataFrame(
    0.0,
    index=res_names,
    columns=patients
)

for j, macro in enumerate(
    res_names
):

    total_weight = M_res[:, j].sum()

    for patient in patients:

        mask = (
            patient_values == patient
        )

        patient_comp_res.loc[
            macro,
            patient
        ] = (
            M_res[mask, j].sum()
            / total_weight
        )

display(
    patient_comp_res.round(4)
)

print(
    "\nRow sums:"
)

display(
    patient_comp_res
    .sum(axis=1)
    .to_frame("Row_sum")
    .round(6)
)

In [ ]:
# =============================================================================
# Step28E Cell 28
# Integrated macrostate summary
# =============================================================================

summary_rows_res = []

for j, macro in enumerate(
    res_names
):

    w = M_res[:, j]

    state_row = (
        state_comp_res
        .loc[macro]
    )

    patient_row = (
        patient_comp_res
        .loc[macro]
    )

    hard_mask = (
        hard_macro_res == macro
    )

    summary_rows_res.append({

        "Macrostate":
            macro,

        "Dominant_state":
            state_row.idxmax(),

        "Dominant_state_fraction":
            float(
                state_row.max()
            ),

        "Dominant_patient":
            patient_row.idxmax(),

        "Dominant_patient_fraction":
            float(
                patient_row.max()
            ),

        "HCC09_HCC10_fraction":
            float(
                patient_row.get(
                    "HCC09",
                    0.0
                )
                +
                patient_row.get(
                    "HCC10",
                    0.0
                )
            ),

        "MS3_fraction":
            float(
                state_row["MS3"]
            ),

        "MS5_fraction":
            float(
                state_row["MS5"]
            ),

        "MS6_fraction":
            float(
                state_row["MS6"]
            ),

        "Axis_MS3_MS5_MS6_fraction":
            float(
                state_row[
                    [
                        "MS3",
                        "MS5",
                        "MS6"
                    ]
                ].sum()
            ),

        "Weighted_mean_PT":
            float(
                np.sum(
                    w * pt
                )
                /
                np.sum(w)
            ),

        "Hard_N":
            int(
                hard_mask.sum()
            ),

        "Mean_confidence":
            float(
                np.mean(
                    max_membership_res[
                        hard_mask
                    ]
                )
            )
    })

summary_res = (
    pd.DataFrame(
        summary_rows_res
    )
    .sort_values(
        "Weighted_mean_PT"
    )
    .reset_index(
        drop=True
    )
)

display(
    summary_res.round(4)
)

In [ ]:
# =============================================================================
# Step28E Cell 29
# Biological program profiles
# =============================================================================

program_res = pd.DataFrame(
    index=res_names,
    columns=program_cols,
    dtype=float
)

for j, macro in enumerate(
    res_names
):

    w = M_res[:, j]

    for col in program_cols:

        x = (
            adata28.obs[col]
            .to_numpy(
                dtype=float
            )
        )

        valid = np.isfinite(
            x
        )

        program_res.loc[
            macro,
            col
        ] = (
            np.sum(
                w[valid]
                * x[valid]
            )
            /
            np.sum(
                w[valid]
            )
        )

display(
    program_res.round(4)
)

In [ ]:
# =============================================================================
# Step28E Cell 30
# Selected-resolution coarse transition matrix
# =============================================================================

coarse_T_res = (
    g_res.coarse_T.copy()
)

display(
    coarse_T_res.round(4)
)

coarse_res_array = (
    coarse_T_res
    .to_numpy()
)

stability_res = pd.Series(
    np.diag(
        coarse_res_array
    ),
    index=coarse_T_res.index,
    name="Stability"
)

print(
    "\nMacrostate stability:"
)

display(
    stability_res
    .sort_values(
        ascending=False
    )
    .to_frame()
    .round(4)
)

In [ ]:
# =============================================================================
# Step28E Cell 31
# Save resolution-sensitivity outputs
# =============================================================================

state_comp_res.to_csv(
    TABLE_DIR /
    "Step28E_MinChi_State_Composition.csv"
)

patient_comp_res.to_csv(
    TABLE_DIR /
    "Step28E_MinChi_Patient_Composition.csv"
)

summary_res.to_csv(
    TABLE_DIR /
    "Step28E_MinChi_Macrostate_Summary.csv",
    index=False
)

program_res.to_csv(
    TABLE_DIR /
    "Step28E_MinChi_Program_Profile.csv"
)

coarse_T_res.to_csv(
    TABLE_DIR /
    "Step28E_MinChi_Coarse_Transition_Matrix.csv"
)

membership_res_df = pd.DataFrame(
    M_res,
    index=adata28.obs_names,
    columns=[
        f"Macrostate_{x}_membership"
        for x in res_names
    ]
)

membership_res_df.insert(
    0,
    "Hard_macrostate",
    hard_macro_res
)

membership_res_df.insert(
    1,
    "Max_membership",
    max_membership_res
)

membership_res_df.insert(
    2,
    "Patient",
    patient_values
)

membership_res_df.insert(
    3,
    "Frozen_state",
    state_values
)

membership_res_df.insert(
    4,
    "Pseudotime",
    pt
)

membership_res_df.to_csv(
    TABLE_DIR /
    "Step28E_MinChi_Cell_Memberships.csv"
)

print(
    "Step28E minChi tables saved."
)

In [ ]:
# =============================================================================
# Step28E Cell 32
# REPORT — Global macrostate resolution sensitivity
# =============================================================================

print(
    "=" * 100
)

print(
    "STEP28E GLOBAL MACROSTATE RESOLUTION REPORT"
)

print(
    "=" * 100
)

print(
    "\nCandidate resolution interval:",
    candidate_range
)

print(
    "minChi-selected number:",
    selected_n_states
)

print(
    "\n[1] Integrated macrostate summary"
)

display(
    summary_res.round(4)
)

print(
    "\n[2] Frozen-state composition"
)

display(
    state_comp_res.round(4)
)

print(
    "\n[3] Patient composition"
)

display(
    patient_comp_res.round(4)
)

print(
    "\n[4] Biological program profile"
)

display(
    program_res.round(4)
)

print(
    "\n[5] Coarse transition matrix"
)

display(
    coarse_T_res.round(4)
)

print(
    "\n[6] Macrostate stability"
)

display(
    stability_res
    .sort_values(
        ascending=False
    )
    .to_frame()
    .round(4)
)

print(
    "\n[7] Membership confidence"
)

print(
    "Median:",
    round(
        float(
            np.median(
                max_membership_res
            )
        ),
        4
    )
)

print(
    "P10:",
    round(
        float(
            np.quantile(
                max_membership_res,
                0.10
            )
        ),
        4
    )
)

print(
    "P90:",
    round(
        float(
            np.quantile(
                max_membership_res,
                0.90
            )
        ),
        4
    )
)

print(
    "\nInterpretation boundary:"
)

print(
    "Resolution was selected by minChi "
    "within the pre-defined interval 3-8."
)

print(
    "Frozen state labels and patient identity "
    "were used only for post-hoc characterization."
)

print(
    "This analysis remains conditioned on the "
    "Step27 pseudotime-defined transition operator."
)

print(
    "\nSTEP28E COMPLETE"
)

In [ ]:
# =============================================================================
# Step28F Cell 33
# Patient-conditioned GPCCA setup
# =============================================================================

import inspect
import scipy.sparse as sp

from scipy.sparse.csgraph import connected_components
from cellrank.kernels import PrecomputedKernel
from cellrank.estimators import GPCCA

print(
    "PrecomputedKernel:",
    inspect.signature(
        PrecomputedKernel
    )
)

print(
    "GPCCA:",
    inspect.signature(
        GPCCA
    )
)

In [ ]:
# =============================================================================
# Step28F Cell 34
# Build patient-conditioned transition operator
# =============================================================================

def build_patient_operator(
    patient_name
):

    mask = (
        patient_values
        == patient_name
    )

    idx = np.flatnonzero(
        mask
    )

    ad_p = (
        adata28[idx]
        .copy()
    )

    T_raw = (
        T_wsl[idx][:, idx]
        .tocsr()
    )

    same_patient_mass = (
        np.asarray(
            T_raw.sum(axis=1)
        )
        .ravel()
    )

    if np.any(
        same_patient_mass <= 0
    ):
        raise RuntimeError(
            f"{patient_name}: "
            "zero within-patient outgoing mass detected."
        )

    inv_mass = (
        1.0 /
        same_patient_mass
    )

    T_cond = (
        sp.diags(inv_mass)
        @ T_raw
    ).tocsr()

    T_cond.eliminate_zeros()

    row_sums = (
        np.asarray(
            T_cond.sum(axis=1)
        )
        .ravel()
    )

    ad_p.obsp[
        "T_patient_conditioned"
    ] = T_cond

    pk_p = PrecomputedKernel(
        ad_p,
        obsp_key="T_patient_conditioned",
        backward=False
    )

    return {
        "adata": ad_p,
        "kernel": pk_p,
        "T_raw": T_raw,
        "T_cond": T_cond,
        "same_mass": same_patient_mass,
        "row_sums": row_sums,
        "indices": idx
    }

In [ ]:
# =============================================================================
# Step28F Cell 35A
# Audit zero within-patient outgoing mass
# =============================================================================

def audit_patient_mass(patient_name):

    mask = (
        patient_values == patient_name
    )

    idx = np.flatnonzero(mask)

    T_raw = (
        T_wsl[idx][:, idx]
        .tocsr()
    )

    same_mass = np.asarray(
        T_raw.sum(axis=1)
    ).ravel()

    zero_mask = (
        same_mass <= 1e-15
    )

    ad_p = adata28[idx]

    audit = pd.DataFrame({
        "Cell_ID":
            ad_p.obs_names,

        "Patient":
            patient_name,

        "State":
            ad_p.obs[STATE_COL]
            .astype(str)
            .to_numpy(),

        "Pseudotime":
            ad_p.obs[PT_KEY]
            .to_numpy(dtype=float),

        "Same_patient_mass":
            same_mass,

        "Zero_same_patient_mass":
            zero_mask
    })

    return {
        "indices": idx,
        "T_raw": T_raw,
        "same_mass": same_mass,
        "zero_mask": zero_mask,
        "audit": audit
    }


audit09 = audit_patient_mass("HCC09")
audit10 = audit_patient_mass("HCC10")

In [ ]:
# =============================================================================
# Step28F Cell 35B
# Zero-mass summary
# =============================================================================

zero_summary = []

for patient, obj in [
    ("HCC09", audit09),
    ("HCC10", audit10)
]:

    z = obj["zero_mask"]

    zero_summary.append({
        "Patient": patient,
        "N_total": len(z),
        "N_zero_mass": int(z.sum()),
        "Zero_fraction": float(z.mean()),
        "N_positive_mass": int((~z).sum()),
        "Min_mass_positive":
            float(
                obj["same_mass"][~z].min()
            ) if (~z).any() else np.nan,

        "Median_mass":
            float(
                np.median(
                    obj["same_mass"]
                )
            )
    })

zero_summary = pd.DataFrame(
    zero_summary
)

display(
    zero_summary.round(6)
)

In [ ]:
# =============================================================================
# Step28F Cell 35C
# State distribution of zero-mass cells
# =============================================================================

zero_state_tables = {}

for patient, obj in [
    ("HCC09", audit09),
    ("HCC10", audit10)
]:

    df = obj["audit"]

    tab = pd.crosstab(
        df["State"],
        df["Zero_same_patient_mass"]
    )

    tab.columns = [
        "Positive_mass"
        if x is False
        else "Zero_mass"
        for x in tab.columns
    ]

    if "Zero_mass" not in tab.columns:
        tab["Zero_mass"] = 0

    tab["Zero_fraction_within_state"] = (
        tab["Zero_mass"] /
        tab.sum(axis=1)
    )

    zero_state_tables[
        patient
    ] = tab

    print(
        f"\n===== {patient} ====="
    )

    display(
        tab.round(4)
    )

In [ ]:
# =============================================================================
# Step28F Cell 35D
# Pseudotime distribution of zero-mass cells
# =============================================================================

zero_pt_rows = []

for patient, obj in [
    ("HCC09", audit09),
    ("HCC10", audit10)
]:

    df = obj["audit"]

    for flag, label in [
        (False, "Positive_mass"),
        (True, "Zero_mass")
    ]:

        x = df.loc[
            df["Zero_same_patient_mass"] == flag,
            "Pseudotime"
        ].to_numpy()

        if len(x) == 0:
            continue

        zero_pt_rows.append({
            "Patient": patient,
            "Group": label,
            "N": len(x),
            "Median_PT": np.median(x),
            "P10_PT": np.quantile(x, 0.10),
            "P90_PT": np.quantile(x, 0.90)
        })

zero_pt_summary = pd.DataFrame(
    zero_pt_rows
)

display(
    zero_pt_summary.round(4)
)

In [ ]:
# =============================================================================
# Step28F Cell 35E
# REPORT — zero within-patient mass audit
# =============================================================================

print("=" * 100)
print("STEP28F ZERO WITHIN-PATIENT MASS AUDIT")
print("=" * 100)

display(
    zero_summary.round(6)
)

for patient in [
    "HCC09",
    "HCC10"
]:

    print(
        f"\n[{patient}] Zero-mass state distribution"
    )

    display(
        zero_state_tables[
            patient
        ].round(4)
    )

print(
    "\nPseudotime distribution:"
)

display(
    zero_pt_summary.round(4)
)

print(
    "\nInterpretation:"
)

print(
    "Zero within-patient mass means that after "
    "removing cross-patient transition edges, "
    "some cells have no remaining outgoing edge."
)

print(
    "No repair, exclusion, self-loop addition, "
    "or graph reconstruction has been performed yet."
)

print(
    "\nSTEP28F AUDIT COMPLETE"
)

In [ ]:
# =============================================================================
# Step28F Cell 35F
# Iteratively remove zero-outgoing cells
# =============================================================================

import scipy.sparse as sp

from cellrank.kernels import PrecomputedKernel


def build_pruned_patient_operator(
    patient_name,
    eps=1e-15
):

    initial_idx = np.flatnonzero(
        patient_values == patient_name
    )

    kept_idx = initial_idx.copy()

    dropped_records = []

    iteration = 0

    while True:

        T_sub = (
            T_wsl[kept_idx][:, kept_idx]
            .tocsr()
        )

        mass = np.asarray(
            T_sub.sum(axis=1)
        ).ravel()

        zero_mask = (
            mass <= eps
        )

        if not zero_mask.any():
            break

        iteration += 1

        drop_idx = (
            kept_idx[zero_mask]
        )

        drop_obs = (
            adata28.obs
            .iloc[drop_idx]
        )

        for cell_id, (_, row) in zip(
            adata28.obs_names[drop_idx],
            drop_obs.iterrows()
        ):

            dropped_records.append({
                "Patient":
                    patient_name,

                "Iteration":
                    iteration,

                "Cell_ID":
                    cell_id,

                "State":
                    str(
                        row[STATE_COL]
                    ),

                "Pseudotime":
                    float(
                        row[PT_KEY]
                    )
            })

        kept_idx = (
            kept_idx[~zero_mask]
        )

        if len(kept_idx) == 0:
            raise RuntimeError(
                f"{patient_name}: "
                "all cells removed."
            )

    # Final patient-only matrix
    T_final_raw = (
        T_wsl[kept_idx][:, kept_idx]
        .tocsr()
    )

    final_mass = np.asarray(
        T_final_raw.sum(axis=1)
    ).ravel()

    assert np.all(
        final_mass > eps
    )

    # Row-normalize
    T_cond = (
        sp.diags(
            1.0 / final_mass
        )
        @ T_final_raw
    ).tocsr()

    T_cond.eliminate_zeros()

    row_sums = np.asarray(
        T_cond.sum(axis=1)
    ).ravel()

    # Subset AnnData
    ad_p = (
        adata28[kept_idx]
        .copy()
    )

    ad_p.obsp[
        "T_patient_conditioned"
    ] = T_cond

    # Feed precomputed transition matrix to CellRank
    pk_p = PrecomputedKernel(
        ad_p,
        obsp_key="T_patient_conditioned",
        backward=False
    )

    dropped_df = pd.DataFrame(
        dropped_records
    )

    return {
        "patient":
            patient_name,

        "initial_idx":
            initial_idx,

        "kept_idx":
            kept_idx,

        "adata":
            ad_p,

        "T_raw":
            T_final_raw,

        "T_cond":
            T_cond,

        "kernel":
            pk_p,

        "final_mass":
            final_mass,

        "row_sums":
            row_sums,

        "dropped":
            dropped_df,

        "n_iterations":
            iteration
    }

In [ ]:
# =============================================================================
# Step28F Cell 35G
# Build pruned patient-conditioned operators
# =============================================================================

p09 = build_pruned_patient_operator(
    "HCC09"
)

p10 = build_pruned_patient_operator(
    "HCC10"
)

for obj in [
    p09,
    p10
]:

    print(
        obj["patient"],
        "Original:",
        len(obj["initial_idx"]),
        "Retained:",
        len(obj["kept_idx"]),
        "Dropped:",
        len(obj["initial_idx"])
        - len(obj["kept_idx"]),
        "Iterations:",
        obj["n_iterations"]
    )

In [ ]:
# =============================================================================
# Step28F Cell 36
# Exclusion audit after iterative pruning
# =============================================================================

exclusion_rows = []

for obj in [
    p09,
    p10
]:

    patient = obj["patient"]

    original_obs = (
        adata28.obs.iloc[
            obj["initial_idx"]
        ]
    )

    retained_obs = (
        obj["adata"].obs
    )

    for state in sorted(
        original_obs[
            STATE_COL
        ].astype(str).unique()
    ):

        n_original = int(
            (
                original_obs[
                    STATE_COL
                ].astype(str)
                == state
            ).sum()
        )

        n_retained = int(
            (
                retained_obs[
                    STATE_COL
                ].astype(str)
                == state
            ).sum()
        )

        exclusion_rows.append({
            "Patient":
                patient,

            "State":
                state,

            "N_original":
                n_original,

            "N_retained":
                n_retained,

            "N_excluded":
                n_original
                - n_retained,

            "Excluded_fraction":
                (
                    n_original
                    - n_retained
                )
                / n_original
        })

exclusion_qc = pd.DataFrame(
    exclusion_rows
)

display(
    exclusion_qc.round(4)
)

In [ ]:
# =============================================================================
# Step28F Cell 37
# Mathematical and graph-connectivity QC
# =============================================================================

from scipy.sparse.csgraph import (
    connected_components
)

connectivity_rows = []

for obj in [
    p09,
    p10
]:

    T = obj["T_cond"]

    # Weak connectivity
    A_weak = (
        T + T.T
    )

    n_weak, weak_labels = (
        connected_components(
            A_weak,
            directed=False
        )
    )

    weak_sizes = np.bincount(
        weak_labels
    )

    # Strong connectivity
    n_strong, strong_labels = (
        connected_components(
            T,
            directed=True,
            connection="strong"
        )
    )

    connectivity_rows.append({
        "Patient":
            obj["patient"],

        "N_cells":
            T.shape[0],

        "N_iterations":
            obj["n_iterations"],

        "N_weak_components":
            int(n_weak),

        "Largest_weak_component_N":
            int(
                weak_sizes.max()
            ),

        "Largest_weak_component_fraction":
            float(
                weak_sizes.max()
                / T.shape[0]
            ),

        "N_strong_components":
            int(n_strong),

        "Max_row_sum_error":
            float(
                np.max(
                    np.abs(
                        obj["row_sums"]
                        - 1.0
                    )
                )
            )
    })

connectivity_qc = pd.DataFrame(
    connectivity_rows
)

display(
    connectivity_qc
)

In [ ]:
# =============================================================================
# Step28F Cell 38
# Patient-conditioned MS6-MS5-MS3 flow
# =============================================================================

axis_pairs = [
    ("MS6", "MS5"),
    ("MS5", "MS3"),
    ("MS6", "MS3")
]

patient_axis_rows = []

for obj in [
    p09,
    p10
]:

    patient = obj["patient"]

    T = obj["T_cond"]

    states_p = (
        obj["adata"].obs[
            STATE_COL
        ]
        .astype(str)
        .to_numpy()
    )

    for source, target in axis_pairs:

        src = np.flatnonzero(
            states_p == source
        )

        dst = np.flatnonzero(
            states_p == target
        )

        rev_src = np.flatnonzero(
            states_p == target
        )

        rev_dst = np.flatnonzero(
            states_p == source
        )

        forward_flow = float(
            T[src][:, dst].sum()
            / len(src)
        )

        reverse_flow = float(
            T[rev_src][:, rev_dst].sum()
            / len(rev_src)
        )

        patient_axis_rows.append({
            "Patient":
                patient,

            "Forward":
                f"{source}->{target}",

            "Forward_flow":
                forward_flow,

            "Reverse":
                f"{target}->{source}",

            "Reverse_flow":
                reverse_flow,

            "Net_forward":
                forward_flow
                - reverse_flow,

            "Forward_reverse_ratio":
                (
                    forward_flow
                    + 1e-12
                )
                /
                (
                    reverse_flow
                    + 1e-12
                )
        })

patient_axis_flow = pd.DataFrame(
    patient_axis_rows
)

display(
    patient_axis_flow.round(4)
)

In [ ]:
# =============================================================================
# Step28F Cell 39
# Formal QC gate before GPCCA
# =============================================================================

ROW_SUM_PASS = (
    connectivity_qc[
        "Max_row_sum_error"
    ].max()
    < 1e-10
)

WEAK_CONNECTIVITY_PASS = (
    connectivity_qc[
        "N_weak_components"
    ].max()
    == 1
)

RETENTION_PASS = (
    (
        connectivity_qc[
            "N_cells"
        ]
        /
        pd.Series(
            [
                len(
                    p09["initial_idx"]
                ),
                len(
                    p10["initial_idx"]
                )
            ]
        )
    ).min()
    > 0.97
)

PATIENT_OPERATOR_PASS = (
    ROW_SUM_PASS
    and
    WEAK_CONNECTIVITY_PASS
    and
    RETENTION_PASS
)

print(
    "ROW_SUM_PASS:",
    ROW_SUM_PASS
)

print(
    "WEAK_CONNECTIVITY_PASS:",
    WEAK_CONNECTIVITY_PASS
)

print(
    "RETENTION_PASS:",
    RETENTION_PASS
)

print(
    "\nPATIENT_OPERATOR_PASS:",
    PATIENT_OPERATOR_PASS
)

In [ ]:
# =============================================================================
# Step28F Cell 40
# REPORT — patient-conditioned operator final QC
# =============================================================================

print("=" * 100)

print(
    "STEP28F PATIENT-CONDITIONED OPERATOR QC REPORT"
)

print("=" * 100)

print(
    "\n[1] Exclusion audit"
)

display(
    exclusion_qc.round(4)
)

print(
    "\n[2] Connectivity QC"
)

display(
    connectivity_qc
)

print(
    "\n[3] Patient-conditioned axis flow"
)

display(
    patient_axis_flow.round(4)
)

print(
    "\n[4] QC gates"
)

print(
    "ROW_SUM_PASS:",
    ROW_SUM_PASS
)

print(
    "WEAK_CONNECTIVITY_PASS:",
    WEAK_CONNECTIVITY_PASS
)

print(
    "RETENTION_PASS:",
    RETENTION_PASS
)

print(
    "PATIENT_OPERATOR_PASS:",
    PATIENT_OPERATOR_PASS
)

print(
    "\nMethodological boundary:"
)

print(
    "Zero-outgoing cells were iteratively excluded; "
    "no artificial self-loops were added."
)

print(
    "No neighbors, pseudotime, malignant-state labels, "
    "or biological programs were re-estimated."
)

print(
    "\nSTEP28F OPERATOR QC COMPLETE"
)

In [ ]:
# =============================================================================
# Step28F Cell 41
# Audit HCC09 weak components
# =============================================================================

T09 = p09["T_cond"]

A09_weak = (
    T09 + T09.T
)

n_comp09, labels09 = connected_components(
    A09_weak,
    directed=False
)

component_sizes09 = np.bincount(
    labels09
)

component_order09 = np.argsort(
    component_sizes09
)[::-1]

print(
    "HCC09 weak components:",
    n_comp09
)

print(
    "Component sizes:",
    component_sizes09
)

print(
    "Largest component:",
    component_order09[0],
    "N =",
    component_sizes09[
        component_order09[0]
    ]
)

In [ ]:
# =============================================================================
# Step28F Cell 42
# HCC09 component composition
# =============================================================================

hcc09_component_df = pd.DataFrame({
    "Cell_ID":
        p09["adata"].obs_names,

    "Component":
        labels09,

    "State":
        p09["adata"].obs[
            STATE_COL
        ].astype(str).to_numpy(),

    "Pseudotime":
        p09["adata"].obs[
            PT_KEY
        ].to_numpy(dtype=float)
})

component_state_tab = pd.crosstab(
    hcc09_component_df[
        "Component"
    ],
    hcc09_component_df[
        "State"
    ]
)

component_state_tab[
    "Total"
] = component_state_tab.sum(
    axis=1
)

component_state_tab = (
    component_state_tab
    .sort_values(
        "Total",
        ascending=False
    )
)

display(
    component_state_tab
)

In [ ]:
# =============================================================================
# Step28F Cell 43
# Largest-component retention audit
# =============================================================================

largest_comp09 = int(
    component_order09[0]
)

lcc_mask09 = (
    labels09 == largest_comp09
)

obs09 = p09["adata"].obs

state09 = (
    obs09[STATE_COL]
    .astype(str)
)

retention_rows09 = []

for state in sorted(
    state09.unique()
):

    state_mask = (
        state09.to_numpy()
        == state
    )

    n_before = int(
        state_mask.sum()
    )

    n_after = int(
        (
            state_mask
            & lcc_mask09
        ).sum()
    )

    retention_rows09.append({
        "State":
            state,

        "N_before_LCC":
            n_before,

        "N_after_LCC":
            n_after,

        "N_removed_LCC":
            n_before - n_after,

        "Retention_fraction":
            n_after / n_before
    })

lcc_retention09 = pd.DataFrame(
    retention_rows09
)

display(
    lcc_retention09.round(4)
)

In [ ]:
# =============================================================================
# Step28F Cell 44
# Pre-defined largest-component QC gate
# =============================================================================

largest_fraction09 = float(
    lcc_mask09.mean()
)

axis_states = [
    "MS3",
    "MS5",
    "MS6"
]

axis_retention09 = (
    lcc_retention09
    .set_index("State")
    .loc[
        axis_states,
        "Retention_fraction"
    ]
)

original_hcc09_n = len(
    p09["initial_idx"]
)

final_hcc09_n_if_lcc = int(
    lcc_mask09.sum()
)

overall_retention09 = (
    final_hcc09_n_if_lcc
    / original_hcc09_n
)

LCC_SIZE_PASS = (
    largest_fraction09 >= 0.99
)

AXIS_RETENTION_PASS = (
    axis_retention09.min()
    >= 0.95
)

OVERALL_RETENTION_PASS = (
    overall_retention09 >= 0.97
)

HCC09_LCC_PASS = (
    LCC_SIZE_PASS
    and
    AXIS_RETENTION_PASS
    and
    OVERALL_RETENTION_PASS
)

print(
    "Largest component fraction:",
    largest_fraction09
)

print(
    "\nAxis-state retention:"
)

display(
    axis_retention09
    .to_frame(
        "Retention_fraction"
    )
)

print(
    "\nOverall retention vs original HCC09:",
    overall_retention09
)

print(
    "\nLCC_SIZE_PASS:",
    LCC_SIZE_PASS
)

print(
    "AXIS_RETENTION_PASS:",
    AXIS_RETENTION_PASS
)

print(
    "OVERALL_RETENTION_PASS:",
    OVERALL_RETENTION_PASS
)

print(
    "\nHCC09_LCC_PASS:",
    HCC09_LCC_PASS
)

In [ ]:
# =============================================================================
# Step28F Cell 45
# Finalize HCC09 largest-component operator
# =============================================================================

assert HCC09_LCC_PASS

ad09_final = (
    p09["adata"][
        lcc_mask09
    ]
    .copy()
)

T09_final = (
    p09["T_cond"][
        lcc_mask09
    ][:, lcc_mask09]
    .tocsr()
)

mass09_final = np.asarray(
    T09_final.sum(axis=1)
).ravel()

assert np.all(
    mass09_final > 0
)

T09_final = (
    sp.diags(
        1.0 / mass09_final
    )
    @ T09_final
).tocsr()

T09_final.eliminate_zeros()

ad09_final.obsp[
    "T_patient_conditioned"
] = T09_final

pk09_final = PrecomputedKernel(
    ad09_final,
    obsp_key="T_patient_conditioned",
    backward=False
)

print(
    "HCC09 final:",
    ad09_final.shape
)

print(
    "Max row-sum error:",
    np.max(
        np.abs(
            np.asarray(
                T09_final.sum(axis=1)
            ).ravel()
            - 1
        )
    )
)

In [ ]:
# =============================================================================
# Step28F Cell 46
# Finalize HCC10 operator
# =============================================================================

ad10_final = p10["adata"]
T10_final = p10["T_cond"]
pk10_final = p10["kernel"]

print(
    "HCC10 final:",
    ad10_final.shape
)

In [ ]:
# =============================================================================
# Step28F Cell 47
# REPORT — final patient-conditioned operators
# =============================================================================

print("=" * 100)

print(
    "STEP28F FINAL PATIENT-OPERATOR REPORT"
)

print("=" * 100)

print(
    "\nHCC09:"
)

print(
    "Original N:",
    original_hcc09_n
)

print(
    "After zero-mass pruning:",
    p09["adata"].n_obs
)

print(
    "Final LCC N:",
    ad09_final.n_obs
)

print(
    "Final retention:",
    ad09_final.n_obs
    / original_hcc09_n
)

print(
    "\nHCC09 axis-state retention:"
)

display(
    axis_retention09
    .to_frame(
        "Retention_fraction"
    )
)

print(
    "\nHCC10:"
)

print(
    "Original N:",
    len(
        p10["initial_idx"]
    )
)

print(
    "Final N:",
    ad10_final.n_obs
)

print(
    "Final retention:",
    ad10_final.n_obs
    /
    len(
        p10["initial_idx"]
    )
)

print(
    "\nHCC09 LCC gate:",
    HCC09_LCC_PASS
)

print(
    "\nNo artificial self-loops added."
)

print(
    "No graph, pseudotime, state labels, "
    "or biological programs re-estimated."
)

print(
    "\nSTEP28F FINAL OPERATORS READY"
)

In [ ]:
# =============================================================================
# Step28G Cell 48
# Save final patient-conditioned operator audit
# =============================================================================

final_operator_summary = pd.DataFrame([
    {
        "Patient": "HCC09",
        "Original_N": 1522,
        "Final_N": ad09_final.n_obs,
        "Retention_fraction": ad09_final.n_obs / 1522,
        "Operator_type": "Largest weak component after iterative zero-mass pruning"
    },
    {
        "Patient": "HCC10",
        "Original_N": 2679,
        "Final_N": ad10_final.n_obs,
        "Retention_fraction": ad10_final.n_obs / 2679,
        "Operator_type": "Iterative zero-mass pruning"
    }
])

final_operator_summary.to_csv(
    TABLE_DIR / "Step28G_Final_Patient_Operator_Summary.csv",
    index=False
)

pd.Series(
    ad09_final.obs_names,
    name="Cell_ID"
).to_csv(
    TABLE_DIR / "Step28G_HCC09_Final_CellIDs.csv",
    index=False
)

pd.Series(
    ad10_final.obs_names,
    name="Cell_ID"
).to_csv(
    TABLE_DIR / "Step28G_HCC10_Final_CellIDs.csv",
    index=False
)

display(final_operator_summary)

In [ ]:
# =============================================================================
# Step28G Cell 49
# HCC09 sparse Schur decomposition
# =============================================================================

from cellrank.estimators import GPCCA

g09 = GPCCA(
    pk09_final
)

g09.compute_schur(
    n_components=12,
    method="krylov",
    which="LR"
)

print(
    "HCC09 Schur vectors:",
    g09.schur_vectors.shape
)

print(
    "HCC09 Schur matrix:",
    g09.schur_matrix.shape
)

In [ ]:
# =============================================================================
# Step28G Cell 50
# HCC10 sparse Schur decomposition
# =============================================================================

g10 = GPCCA(
    pk10_final
)

g10.compute_schur(
    n_components=12,
    method="krylov",
    which="LR"
)

print(
    "HCC10 Schur vectors:",
    g10.schur_vectors.shape
)

print(
    "HCC10 Schur matrix:",
    g10.schur_matrix.shape
)

In [ ]:
# =============================================================================
# Step28G Cell 51
# Helper for patient-specific eigenspectra
# =============================================================================

def extract_patient_spectrum(
    estimator,
    patient
):

    eig = estimator.eigendecomposition

    vals = np.asarray(
        eig["D"]
    )

    if vals.ndim == 2:
        vals = np.diag(vals)

    table = pd.DataFrame({
        "Patient":
            patient,

        "Component":
            np.arange(
                1,
                len(vals) + 1
            ),

        "Real":
            np.real(vals),

        "Imag":
            np.imag(vals),

        "Magnitude":
            np.abs(vals)
    })

    return (
        table,
        eig.get("eigengap")
    )

In [ ]:
# =============================================================================
# Step28G Cell 52
# Patient-specific eigenspectra
# =============================================================================

eig09, gap09 = extract_patient_spectrum(
    g09,
    "HCC09"
)

eig10, gap10 = extract_patient_spectrum(
    g10,
    "HCC10"
)

print(
    "HCC09 eigengap:",
    gap09
)

display(
    eig09.round(6)
)

print(
    "\nHCC10 eigengap:",
    gap10
)

display(
    eig10.round(6)
)

In [ ]:
# =============================================================================
# Step28G Cell 53
# HCC09 eigengap-driven primary macrostates
# =============================================================================

g09.compute_macrostates(
    n_states=None,
    n_cells=30,
    cluster_key=None
)

M09 = np.asarray(
    g09.macrostates_memberships
)

names09 = list(
    g09.macrostates_memberships.names
)

print(
    "HCC09 primary macrostates:",
    len(names09)
)

print(
    "Names:",
    names09
)

In [ ]:
# =============================================================================
# Step28G Cell 54
# HCC10 eigengap-driven primary macrostates
# =============================================================================

g10.compute_macrostates(
    n_states=None,
    n_cells=30,
    cluster_key=None
)

M10 = np.asarray(
    g10.macrostates_memberships
)

names10 = list(
    g10.macrostates_memberships.names
)

print(
    "HCC10 primary macrostates:",
    len(names10)
)

print(
    "Names:",
    names10
)

In [ ]:
# =============================================================================
# Step28G Cell 55
# Characterize patient-specific macrostates
# =============================================================================

def summarize_patient_macrostates(
    adata_p,
    memberships,
    names,
    estimator,
    patient
):

    M = memberships

    state_p = (
        adata_p.obs[STATE_COL]
        .astype(str)
        .to_numpy()
    )

    pt_p = (
        adata_p.obs[PT_KEY]
        .to_numpy(dtype=float)
    )

    hard_idx = np.argmax(
        M,
        axis=1
    )

    hard_macro = np.asarray(
        names,
        dtype=object
    )[hard_idx]

    max_conf = np.max(
        M,
        axis=1
    )

    state_list = sorted(
        np.unique(state_p)
    )

    state_comp = pd.DataFrame(
        0.0,
        index=names,
        columns=state_list
    )

    rows = []

    for j, macro in enumerate(names):

        w = M[:, j]

        total = w.sum()

        for state in state_list:

            mask = (
                state_p == state
            )

            state_comp.loc[
                macro,
                state
            ] = (
                w[mask].sum()
                / total
            )

        hard_mask = (
            hard_macro == macro
        )

        rows.append({
            "Patient":
                patient,

            "Macrostate":
                macro,

            "Dominant_state":
                state_comp.loc[
                    macro
                ].idxmax(),

            "Dominant_state_fraction":
                float(
                    state_comp.loc[
                        macro
                    ].max()
                ),

            "MS3_fraction":
                float(
                    state_comp.loc[
                        macro
                    ].get(
                        "MS3",
                        0.0
                    )
                ),

            "MS5_fraction":
                float(
                    state_comp.loc[
                        macro
                    ].get(
                        "MS5",
                        0.0
                    )
                ),

            "MS6_fraction":
                float(
                    state_comp.loc[
                        macro
                    ].get(
                        "MS6",
                        0.0
                    )
                ),

            "Axis_fraction":
                float(
                    sum(
                        state_comp.loc[
                            macro
                        ].get(
                            s,
                            0.0
                        )
                        for s in [
                            "MS3",
                            "MS5",
                            "MS6"
                        ]
                    )
                ),

            "Weighted_mean_PT":
                float(
                    np.sum(
                        w * pt_p
                    )
                    / total
                ),

            "Hard_N":
                int(
                    hard_mask.sum()
                ),

            "Mean_confidence":
                float(
                    max_conf[
                        hard_mask
                    ].mean()
                )
        })

    summary = pd.DataFrame(
        rows
    ).sort_values(
        "Weighted_mean_PT"
    ).reset_index(
        drop=True
    )

    coarse = (
        estimator
        .coarse_T
        .copy()
    )

    stability = pd.Series(
        np.diag(
            coarse.to_numpy()
        ),
        index=coarse.index,
        name="Stability"
    )

    summary[
        "Stability"
    ] = summary[
        "Macrostate"
    ].map(
        stability
    )

    return {
        "summary":
            summary,

        "state_comp":
            state_comp,

        "coarse_T":
            coarse,

        "stability":
            stability,

        "hard_macro":
            hard_macro,

        "confidence":
            max_conf
    }

In [ ]:
# =============================================================================
# Step28G Cell 56
# HCC09 macrostate characterization
# =============================================================================

res09 = summarize_patient_macrostates(
    ad09_final,
    M09,
    names09,
    g09,
    "HCC09"
)

display(
    res09["summary"].round(4)
)

print(
    "\nState composition:"
)

display(
    res09[
        "state_comp"
    ].round(4)
)

print(
    "\nCoarse transition matrix:"
)

display(
    res09[
        "coarse_T"
    ].round(4)
)

In [ ]:
# =============================================================================
# Step28G Cell 57
# HCC10 macrostate characterization
# =============================================================================

res10 = summarize_patient_macrostates(
    ad10_final,
    M10,
    names10,
    g10,
    "HCC10"
)

display(
    res10["summary"].round(4)
)

print(
    "\nState composition:"
)

display(
    res10[
        "state_comp"
    ].round(4)
)

print(
    "\nCoarse transition matrix:"
)

display(
    res10[
        "coarse_T"
    ].round(4)
)

In [ ]:
# =============================================================================
# Step28G Cell 58
# Biological programs by patient-specific macrostate
# =============================================================================

program_cols_patient = [
    "MS3_conserved_Core50_score",
    "Program__Proliferation_G2M",
    "DeepProgram__Mitotic_Entry",
    "DeepProgram__Spindle_Chromosome_Segregation",
    "DeepProgram__Hepatic_Differentiation",
    "DeepProgram__Bile_Acid_Metabolism"
]


def macro_program_profile(
    adata_p,
    M,
    names
):

    out = pd.DataFrame(
        index=names,
        columns=program_cols_patient,
        dtype=float
    )

    for j, macro in enumerate(
        names
    ):

        w = M[:, j]

        for col in (
            program_cols_patient
        ):

            x = (
                adata_p.obs[col]
                .to_numpy(
                    dtype=float
                )
            )

            valid = np.isfinite(
                x
            )

            out.loc[
                macro,
                col
            ] = (
                np.sum(
                    w[valid]
                    * x[valid]
                )
                /
                np.sum(
                    w[valid]
                )
            )

    return out


prog09 = macro_program_profile(
    ad09_final,
    M09,
    names09
)

prog10 = macro_program_profile(
    ad10_final,
    M10,
    names10
)

print(
    "HCC09"
)

display(
    prog09.round(4)
)

print(
    "\nHCC10"
)

display(
    prog10.round(4)
)

In [ ]:
# =============================================================================
# Step28G Cell 59
# Save patient-specific GPCCA outputs
# =============================================================================

eig09.to_csv(
    TABLE_DIR /
    "Step28G_HCC09_Schur_Eigenvalues.csv",
    index=False
)

eig10.to_csv(
    TABLE_DIR /
    "Step28G_HCC10_Schur_Eigenvalues.csv",
    index=False
)

res09["summary"].to_csv(
    TABLE_DIR /
    "Step28G_HCC09_Primary_Macrostate_Summary.csv",
    index=False
)

res10["summary"].to_csv(
    TABLE_DIR /
    "Step28G_HCC10_Primary_Macrostate_Summary.csv",
    index=False
)

res09["state_comp"].to_csv(
    TABLE_DIR /
    "Step28G_HCC09_State_Composition.csv"
)

res10["state_comp"].to_csv(
    TABLE_DIR /
    "Step28G_HCC10_State_Composition.csv"
)

prog09.to_csv(
    TABLE_DIR /
    "Step28G_HCC09_Program_Profile.csv"
)

prog10.to_csv(
    TABLE_DIR /
    "Step28G_HCC10_Program_Profile.csv"
)

res09["coarse_T"].to_csv(
    TABLE_DIR /
    "Step28G_HCC09_Coarse_Transition_Matrix.csv"
)

res10["coarse_T"].to_csv(
    TABLE_DIR /
    "Step28G_HCC10_Coarse_Transition_Matrix.csv"
)

print(
    "Step28G outputs saved."
)

In [ ]:
# =============================================================================
# Step28G Cell 60
# REPORT — patient-specific primary GPCCA
# =============================================================================

print("=" * 100)
print("STEP28G PATIENT-SPECIFIC PRIMARY GPCCA REPORT")
print("=" * 100)

print(
    "\n[1] HCC09 eigenspectrum"
)

print(
    "Eigengap:",
    gap09
)

display(
    eig09.round(6)
)

print(
    "\nHCC09 primary macrostates:",
    len(names09)
)

display(
    res09[
        "summary"
    ].round(4)
)

print(
    "\nHCC09 state composition"
)

display(
    res09[
        "state_comp"
    ].round(4)
)

print(
    "\nHCC09 biological programs"
)

display(
    prog09.round(4)
)

print(
    "\nHCC09 coarse transition matrix"
)

display(
    res09[
        "coarse_T"
    ].round(4)
)

print(
    "\n" + "-" * 100
)

print(
    "\n[2] HCC10 eigenspectrum"
)

print(
    "Eigengap:",
    gap10
)

display(
    eig10.round(6)
)

print(
    "\nHCC10 primary macrostates:",
    len(names10)
)

display(
    res10[
        "summary"
    ].round(4)
)

print(
    "\nHCC10 state composition"
)

display(
    res10[
        "state_comp"
    ].round(4)
)

print(
    "\nHCC10 biological programs"
)

display(
    prog10.round(4)
)

print(
    "\nHCC10 coarse transition matrix"
)

display(
    res10[
        "coarse_T"
    ].round(4)
)

print(
    "\nInterpretation boundary:"
)

print(
    "Patient-specific macrostates were inferred "
    "from patient-conditioned versions of the "
    "frozen Step28 transition operator."
)

print(
    "No state labels or biological programs "
    "were used to select macrostate number."
)

print(
    "Terminal states and fate probabilities "
    "have NOT yet been computed."
)

print(
    "\nSTEP28G COMPLETE"
)

In [ ]:
# =============================================================================
# Step28H Cell 61
# Audit the tiny HCC10 hard macrostate
# =============================================================================

hard10 = np.asarray(
    res10["hard_macro"],
    dtype=object
)

hard_counts10 = (
    pd.Series(hard10)
    .value_counts()
)

display(
    hard_counts10
    .to_frame("Hard_N")
)

tiny_macro10 = (
    hard_counts10.idxmin()
)

tiny_mask10 = (
    hard10 == tiny_macro10
)

tiny10 = pd.DataFrame({
    "Cell_ID":
        ad10_final.obs_names[
            tiny_mask10
        ],

    "State":
        ad10_final.obs.loc[
            tiny_mask10,
            STATE_COL
        ].astype(str).to_numpy(),

    "Pseudotime":
        ad10_final.obs.loc[
            tiny_mask10,
            PT_KEY
        ].to_numpy(dtype=float)
})

for col in program_cols_patient:

    tiny10[col] = (
        ad10_final.obs.loc[
            tiny_mask10,
            col
        ].to_numpy(dtype=float)
    )

print(
    "Tiny HCC10 macrostate:",
    tiny_macro10
)

display(
    tiny10.round(4)
)

In [ ]:
# =============================================================================
# Step28H Cell 62
# HCC10 strongly connected component audit
# =============================================================================

from scipy.sparse.csgraph import (
    connected_components
)

n_scc10, scc10 = connected_components(
    T10_final,
    directed=True,
    connection="strong"
)

scc_sizes10 = np.bincount(
    scc10
)

Tcoo = T10_final.tocoo()

src_scc = scc10[
    Tcoo.row
]

dst_scc = scc10[
    Tcoo.col
]

cross = (
    src_scc != dst_scc
)

scc_with_outgoing = set(
    src_scc[cross]
)

scc_table10 = pd.DataFrame({
    "SCC":
        np.arange(
            n_scc10
        ),

    "N_cells":
        scc_sizes10
})

scc_table10[
    "Is_sink_SCC"
] = ~scc_table10[
    "SCC"
].isin(
    scc_with_outgoing
)

display(
    scc_table10
    .sort_values(
        "N_cells",
        ascending=False
    )
)

tiny10[
    "SCC"
] = scc10[
    tiny_mask10
]

print(
    "\nTiny-macrostate cells:"
)

display(
    tiny10.round(4)
)

In [ ]:
# =============================================================================
# Step28H Cell 63
# Macrostate-SCC relationship in HCC10
# =============================================================================

macro_scc10 = pd.crosstab(
    pd.Series(
        hard10,
        name="Hard_macrostate"
    ),
    pd.Series(
        scc10,
        name="Strong_component"
    )
)

display(
    macro_scc10
)

In [ ]:
# =============================================================================
# Step28H Cell 64
# Quantify HCC09 macrostate separation
# =============================================================================

h09 = (
    res09["summary"]
    .set_index("Macrostate")
)

p09_prog = prog09.copy()

high_core09 = (
    p09_prog[
        "MS3_conserved_Core50_score"
    ].idxmax()
)

low_core09 = (
    p09_prog[
        "MS3_conserved_Core50_score"
    ].idxmin()
)

hcc09_contrast = pd.DataFrame({
    "Metric": [
        "Weighted_mean_PT",
        "MS3_fraction",
        "MS5_fraction",
        "MS6_fraction",
        "Core50",
        "G2M",
        "Mitotic",
        "Spindle",
        "Hepatic"
    ],

    "High_Core50_macrostate": [
        h09.loc[
            high_core09,
            "Weighted_mean_PT"
        ],
        h09.loc[
            high_core09,
            "MS3_fraction"
        ],
        h09.loc[
            high_core09,
            "MS5_fraction"
        ],
        h09.loc[
            high_core09,
            "MS6_fraction"
        ],
        p09_prog.loc[
            high_core09,
            "MS3_conserved_Core50_score"
        ],
        p09_prog.loc[
            high_core09,
            "Program__Proliferation_G2M"
        ],
        p09_prog.loc[
            high_core09,
            "DeepProgram__Mitotic_Entry"
        ],
        p09_prog.loc[
            high_core09,
            "DeepProgram__Spindle_Chromosome_Segregation"
        ],
        p09_prog.loc[
            high_core09,
            "DeepProgram__Hepatic_Differentiation"
        ]
    ],

    "Low_Core50_macrostate": [
        h09.loc[
            low_core09,
            "Weighted_mean_PT"
        ],
        h09.loc[
            low_core09,
            "MS3_fraction"
        ],
        h09.loc[
            low_core09,
            "MS5_fraction"
        ],
        h09.loc[
            low_core09,
            "MS6_fraction"
        ],
        p09_prog.loc[
            low_core09,
            "MS3_conserved_Core50_score"
        ],
        p09_prog.loc[
            low_core09,
            "Program__Proliferation_G2M"
        ],
        p09_prog.loc[
            low_core09,
            "DeepProgram__Mitotic_Entry"
        ],
        p09_prog.loc[
            low_core09,
            "DeepProgram__Spindle_Chromosome_Segregation"
        ],
        p09_prog.loc[
            low_core09,
            "DeepProgram__Hepatic_Differentiation"
        ]
    ]
})

hcc09_contrast[
    "Difference"
] = (
    hcc09_contrast[
        "High_Core50_macrostate"
    ]
    -
    hcc09_contrast[
        "Low_Core50_macrostate"
    ]
)

display(
    hcc09_contrast.round(4)
)

In [ ]:
# =============================================================================
# Step28H Cell 65
# Final Step28 evidence synthesis
# =============================================================================

step28_evidence = pd.DataFrame([
    {
        "Analysis":
            "Global PseudotimeKernel",

        "Result":
            "Forward-biased globally; "
            "MS6->MS5 robust; "
            "MS5->MS3 not robust",

        "Supports_MS3_terminal_fate":
            False
    },

    {
        "Analysis":
            "Global GPCCA",

        "Result":
            "3 macrostates selected by both "
            "eigengap and minChi; "
            "patient/branch dominated",

        "Supports_MS3_terminal_fate":
            False
    },

    {
        "Analysis":
            "HCC09 patient-conditioned GPCCA",

        "Result":
            "2 mixed MS5/MS6 macrostates; "
            "no clear pseudotime-separated "
            "MS3 terminal region",

        "Supports_MS3_terminal_fate":
            False
    },

    {
        "Analysis":
            "HCC10 patient-conditioned GPCCA",

        "Result":
            "2-state solution with "
            "2654-vs-2 hard assignment; "
            "degenerate metastable structure",

        "Supports_MS3_terminal_fate":
            False
    }
])

display(
    step28_evidence
)

FATE_ANALYSIS_ELIGIBLE = False

print(
    "\nFATE_ANALYSIS_ELIGIBLE:",
    FATE_ANALYSIS_ELIGIBLE
)

In [ ]:
# =============================================================================
# Step28H Cell 66
# Save Step28 Final Lock
# =============================================================================

step28_evidence.to_csv(
    TABLE_DIR /
    "Step28H_Final_Evidence_Synthesis.csv",
    index=False
)

tiny10.to_csv(
    TABLE_DIR /
    "Step28H_HCC10_Tiny_Macrostate_Audit.csv",
    index=False
)

scc_table10.to_csv(
    TABLE_DIR /
    "Step28H_HCC10_Strong_Component_Audit.csv",
    index=False
)

macro_scc10.to_csv(
    TABLE_DIR /
    "Step28H_HCC10_Macrostate_SCC_Crosstab.csv"
)

hcc09_contrast.to_csv(
    TABLE_DIR /
    "Step28H_HCC09_Macrostate_Contrast.csv",
    index=False
)

print(
    "Step28 Final Lock tables saved."
)

In [ ]:
# =============================================================================
# Step28H Cell 67
# FINAL REPORT — Step28 freeze
# =============================================================================

print("=" * 100)
print("STEP28 FINAL LOCK REPORT")
print("=" * 100)

print(
    "\nRNA velocity available: NO"
)

print(
    "\nGlobal directional result:"
)

print(
    "MS6->MS5 is the most reproducible "
    "directional relationship."
)

print(
    "Direct MS5->MS3 one-step flow is "
    "not robust and is reverse-biased "
    "under the primary patient-aware analyses."
)

print(
    "\nGlobal GPCCA:"
)

print(
    "Eigengap and minChi both selected "
    "3 macrostates."
)

print(
    "Macrostates were dominated by "
    "patient/branch-level structure."
)

print(
    "\nHCC09 patient-conditioned GPCCA:"
)

print(
    "Two mixed macrostates were identified."
)

print(
    "The Core50/MS3-enriched macrostate "
    "was not cleanly later in pseudotime "
    "and was not an MS3-dominant endpoint."
)

print(
    "\nHCC10 patient-conditioned GPCCA:"
)

print(
    "Two-state solution was highly asymmetric "
    "in hard assignment (2654 vs 2 cells) "
    "and was not biologically suitable "
    "for MS3 fate interpretation."
)

print(
    "\nTerminal-state / fate-probability analysis:"
)

print(
    "NOT PERFORMED."
)

print(
    "Reason: no reproducible, biologically "
    "interpretable MS3 terminal/metastable "
    "state was identified."
)

print(
    "\nFinal Step28 interpretation:"
)

print(
    "Step28 supports a pseudotime-conditioned "
    "directional departure from the "
    "MS6 differentiated region toward an "
    "MS5-centered transition region."
)

print(
    "The MS5-MS3 interface is locally "
    "bidirectional/complex, and metastable "
    "fate structure is patient-dependent."
)

print(
    "These results do not constitute "
    "independent temporal or lineage proof."
)

print(
    "\nFATE_ANALYSIS_ELIGIBLE:",
    FATE_ANALYSIS_ELIGIBLE
)

print(
    "\nSTEP28 FROZEN — READY FOR STEP29 GRN"
)